# Задача

Представим данные в виде двудольного графа: узлы - пользователи и фильмы, ребра - оценки.  
Задача - обучить эмбеддинги узлов так, чтобы похожие пользователи и фильмы оказались рядом в векторном пространстве.

# Импорт библиотек

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import vstack, hstack, csr_matrix, diags

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется {device}')

Используется cuda


# Предобработка данных

In [33]:
def filter_rare_users_items(ratings, min_item_ratings=5):
    """Удаляет фильмы с малым числом оценок"""
    item_counts = ratings['movieId'].value_counts()
    items_to_keep = item_counts[item_counts >= min_item_ratings].index
    ratings = ratings[ratings['movieId'].isin(items_to_keep)]
    
    return ratings

In [34]:
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')
ratings_filtered = filter_rare_users_items(ratings)
print(len(ratings_filtered))

90274


In [35]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [37]:
# Переиндексация
user_ids = ratings_filtered['userId'].unique()
user_id2idx = {id: idx for idx, id in enumerate(user_ids)}
user_idx2id = {idx: id for id, idx in user_id2idx.items()}

item_ids = ratings_filtered['movieId'].unique()
item_id2idx = {id: idx for idx, id in enumerate(item_ids)}
item_idx2id = {idx: id for id, idx in item_id2idx.items()}

n_users = len(user_ids)
n_items = len(item_ids)

print(f'Пользователей: {n_users}')
print(f'Фильмов: {n_items}')

Пользователей: 610
Фильмов: 3650


In [38]:
def time_train_test_split(df, train_size=0.7, val_size=0.1):
    df.rename(columns={'userId': 'user_id', 'movieId': 'item_id'}, inplace=True)
    df.drop_duplicates(subset=['user_id', 'item_id', 'timestamp'], inplace=True)
    
    df_sorted = df.sort_values(by='timestamp')
    df_sorted['userId'] = df_sorted['user_id']
    grouped = df_sorted.groupby('userId')

    def split_train_test(group):
        train_size_ = int(train_size * len(group))
        val_size_ = int((train_size + val_size) * len(group))
        return group.iloc[:train_size_], group.iloc[train_size_:val_size_], group.iloc[val_size_:]

    train, val, test = zip(*grouped.apply(split_train_test))

    train = pd.concat(train)
    val = pd.concat(val)
    test = pd.concat(test)

    return train, val, test

In [39]:
# Сплит данных
train_df, val_df, test_df = time_train_test_split(ratings_filtered)

print('Размеры выборок:')
print(f'train: {len(train_df)}')
print(f'val: {len(val_df)}')
print(f'test: {len(test_df)}')

Размеры выборок:
train: 62927
val: 8943
test: 18404


In [43]:
train_df['user_idx'] = train_df['user_id'].map(user_id2idx)
train_df['item_idx'] = train_df['item_id'].map(item_id2idx)

val_df['user_idx'] = val_df['user_id'].map(user_id2idx)
val_df['item_idx'] = val_df['item_id'].map(item_id2idx)

test_df['user_idx'] = test_df['user_id'].map(user_id2idx)
test_df['item_idx'] = test_df['item_id'].map(item_id2idx)

# Функция потерь - BPR Loss

In [44]:
def bpr_loss(user_emb, item_emb, pos_items, neg_items):
    user_emb = user_emb[pos_items[:, 0]]
    pos_emb = item_emb[pos_items[:, 1]]
    neg_emb = item_emb[neg_items[:, 1]]

    pos_scores = (user_emb * pos_emb).sum(dim=1)
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    
    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
    return loss

In [45]:
def compute_val_loss(model, val_dataloader, adj_norm, device):
    """
    Вычисляет средний loss на валидации
    """
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        user_emb, item_emb = model(adj_norm)
        
        for batch_users, batch_pos, batch_neg in val_dataloader:
            batch_users = batch_users.to(device)
            batch_pos = batch_pos.to(device)
            batch_neg = batch_neg.to(device)
            
            loss = bpr_loss(
                user_emb, item_emb,
                torch.stack([batch_users, batch_pos], dim=1),
                torch.stack([batch_users, batch_neg], dim=1)
            )
            
            total_loss += loss.item()
    
    return total_loss / len(val_dataloader)

# Матрица смежности

In [46]:
def build_adj_matrix(train_data, n_users, n_items):
    rows = train_data['user_idx'].values
    cols = train_data['item_idx'].values
    data = np.ones(len(train_data))
    
    U = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))
    V = U.T
    
    top = hstack([csr_matrix((n_users, n_users)), U])
    bottom = hstack([V, csr_matrix((n_items, n_items))])
    adj = vstack([top, bottom])
    
    # Нормализация
    deg = np.array(adj.sum(axis=1)).flatten()
    deg[deg == 0] = 1
    deg_sqrt_inv = diags(1.0 / np.sqrt(deg))
    adj_norm = deg_sqrt_inv @ adj @ deg_sqrt_inv
    
    return adj_norm.tocoo()

# Граф на train
adj_norm = build_adj_matrix(train_df, n_users, n_items)

# LightGCN

In [47]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=3):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.emb_dim = emb_dim
        self.n_layers = n_layers
        
        # Начальные эмбеддинги
        self.user_embedding = nn.Embedding(n_users, emb_dim)
        self.item_embedding = nn.Embedding(n_items, emb_dim)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        
    def forward(self, adj_norm):
        ego_embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        all_embeddings = [ego_embeddings]
        
        for _ in range(self.n_layers):
            ego_embeddings = torch.sparse.mm(adj_norm, ego_embeddings)
            all_embeddings.append(ego_embeddings)
        
        final_embeddings = torch.stack(all_embeddings, dim=1).sum(dim=1)
        
        user_emb, item_emb = torch.split(final_embeddings, [self.n_users, self.n_items])
        
        return user_emb, item_emb

In [48]:
class BPRDataset(Dataset):
    def __init__(self, train_data, n_items):
        self.train_data = train_data
        self.n_items = n_items
        self.users = train_data['user_idx'].values
        self.pos_items = train_data['item_idx'].values
        
        self.user_pos_items = {}
        for user, item in zip(self.users, self.pos_items):
            if user not in self.user_pos_items:
                self.user_pos_items[user] = []
            self.user_pos_items[user].append(item)
        
    def __len__(self):
        return len(self.train_data)
    
    def __getitem__(self, idx):
        user = self.users[idx]
        pos_item = self.pos_items[idx]
        
        pos_set = set(self.user_pos_items[user])
        
        # Сэмплируем негативный пример
        while True:
            neg_item = np.random.randint(0, self.n_items)
            if neg_item not in pos_set:
                break
        
        return user, pos_item, neg_item

# Обучение модели

In [49]:
def train_lightgcn(model, train_loader, val_loader, adj_norm, device, epochs=50, lr=1e-3, patience=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    adj_norm = adj_norm.to(device)
    
    history = {
        'train_loss': [],
        'val_loss': []
    }
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(1, epochs + 1):
        # train
        model.train()
        total_train_loss = 0
        
        for batch_users, batch_pos, batch_neg in train_loader:
            batch_users = batch_users.to(device)
            batch_pos = batch_pos.to(device)
            batch_neg = batch_neg.to(device)
            
            user_emb, item_emb = model(adj_norm)
            
            loss = bpr_loss(
                user_emb, item_emb,
                torch.stack([batch_users, batch_pos], dim=1),
                torch.stack([batch_users, batch_neg], dim=1)
            )
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # validation
        avg_val_loss = compute_val_loss(model, val_loader, adj_norm, device)
        history['val_loss'].append(avg_val_loss)
        
        print(f'Epoch {epoch}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss:   {avg_val_loss:.4f}')
        
        # Ранняя остановка
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            
            if patience_counter >= patience:
                print(f'  Early stopping!')
                break
    
    # Восстанавливаем лучшую модель
    model.load_state_dict(best_model_state)
    model.to(device)
    
    print(f'\nОбучение завершено!')
    print(f'Лучший Val Loss: {best_val_loss:.4f}')
    
    return model, history

In [50]:
train_dataset = BPRDataset(train_df, n_items)
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)

val_dataset = BPRDataset(val_df, n_items)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)

test_dataset = BPRDataset(test_df, n_items)
test_dataloader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [80]:
model = LightGCN(n_users, n_items, emb_dim=64, n_layers=3)
model.to(device)

LightGCN(
  (user_embedding): Embedding(610, 64)
  (item_embedding): Embedding(3650, 64)
)

In [86]:
adj_norm_tensor = torch.sparse_coo_tensor(
    np.vstack([adj_norm.row, adj_norm.col]),
    adj_norm.data,
    torch.Size([adj_norm.shape[0], adj_norm.shape[1]]),
    device=device
).to(torch.float32)

In [87]:
model, history = train_lightgcn(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    adj_norm=adj_norm_tensor,
    device=device,
    epochs=50,
    lr=1e-3,
    patience=10
)

Epoch 1/50, Train Loss: 0.5050, Val Loss:   0.6021
Epoch 2/50, Train Loss: 0.3410, Val Loss:   0.5971
Epoch 3/50, Train Loss: 0.2991, Val Loss:   0.5963
Epoch 4/50, Train Loss: 0.2797, Val Loss:   0.6115
Epoch 5/50, Train Loss: 0.2619, Val Loss:   0.6072
Epoch 6/50, Train Loss: 0.2492, Val Loss:   0.6144
Epoch 7/50, Train Loss: 0.2338, Val Loss:   0.5942
Epoch 8/50, Train Loss: 0.2201, Val Loss:   0.5821
Epoch 9/50, Train Loss: 0.2061, Val Loss:   0.5902
Epoch 10/50, Train Loss: 0.1915, Val Loss:   0.5912
Epoch 11/50, Train Loss: 0.1809, Val Loss:   0.6038
Epoch 12/50, Train Loss: 0.1677, Val Loss:   0.6157
Epoch 13/50, Train Loss: 0.1586, Val Loss:   0.6138
Epoch 14/50, Train Loss: 0.1523, Val Loss:   0.5842
Epoch 15/50, Train Loss: 0.1427, Val Loss:   0.6042
Epoch 16/50, Train Loss: 0.1365, Val Loss:   0.6298
Epoch 17/50, Train Loss: 0.1307, Val Loss:   0.6317
Epoch 18/50, Train Loss: 0.1210, Val Loss:   0.6465
  Early stopping!

Обучение завершено!
Лучший Val Loss: 0.5821


# Оценка на тесте

In [88]:
def predict_lightgcn(model, user_ids, movie_ids, adj_norm, device):
    """
    Предсказание рейтингов LightGCN
    """
    model.eval()
    with torch.no_grad():
        user_emb, item_emb = model(adj_norm)
        
        user_emb_selected = user_emb[user_ids]
        item_emb_selected = item_emb[movie_ids]
        
        predictions = (user_emb_selected * item_emb_selected).sum(dim=1)
        
        # Масштабируем в диапазон рейтингов (0.5-5.0)
        predictions = torch.sigmoid(predictions) * 4.5 + 0.5
        
        return predictions.cpu().numpy()

In [89]:
test_users = torch.tensor(test_df['user_idx'].values, device=device)
test_items = torch.tensor(test_df['item_idx'].values, device=device)

pred_ratings = predict_lightgcn(model, test_users, test_items, adj_norm_tensor, device)

In [90]:
true_ratings = test_df['rating'].values
rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))
mae = mean_absolute_error(true_ratings, pred_ratings)

print(f"LightGCN RMSE: {rmse:.4f}")
print(f"LightGCN MAE: {mae:.4f}")

LightGCN RMSE: 1.5302
LightGCN MAE: 1.2034


In [91]:
def evaluate_lightgcn_ranking(model, test_df, adj_norm, device, k=50):
    """
    Оценка LightGCN с помощью метрик ранжирования: Recall@K, NDCG@K
    """
    model.eval()
    
    with torch.no_grad():
        user_emb, item_emb = model(adj_norm)
        
        scores = user_emb @ item_emb.T
        scores = torch.sigmoid(scores)
        
        scores = scores.cpu().numpy()
    
    # Для каждого пользователя в тесте
    recall_list = []
    ndcg_list = []
    
    for user_idx in test_df['user_idx'].unique():        
        # Релевантные фильмы из test (оценки >= 4.0)
        relevant_items = test_df[
            (test_df['user_idx'] == user_idx) & 
            (test_df['rating'] >= 4.0)
        ]['item_idx'].values
        
        if len(relevant_items) == 0:
            continue
        
        user_scores = scores[user_idx].copy()
        
        top_k_indices = np.argsort(-user_scores)[:k]
        recommended_set = set(top_k_indices)
        relevant_set = set(relevant_items)
        
        # Recall@K
        recall = len(relevant_set & recommended_set) / len(relevant_set)
        recall_list.append(recall)
        
        # NDCG@K
        dcg = 0
        for i, item in enumerate(top_k_indices):
            if item in relevant_set:
                dcg += 1 / np.log2(i + 2)
        
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_items), k)))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg)
    
    return {
        f'Recall@{k}': np.mean(recall_list),
        f'NDCG@{k}': np.mean(ndcg_list)
    }

In [92]:
ranking_metrics = evaluate_lightgcn_ranking(model, test_df, adj_norm_tensor, device, k=50)

for key in ranking_metrics.keys():
    print(f'{key}: {ranking_metrics[key]:.4f}')

Recall@50: 0.1434
NDCG@50: 0.0708


In [94]:
# Сохранение модели
torch.save({
    'model_state_dict': model.state_dict(),
    'metrics': ranking_metrics
}, '../models/lightgcn.pt')